# Mashtots — классификация армянских рукописных букв

Локальный прогон CNN на 78 классов для соревнования
[Mashtots Dataset](https://www.kaggle.com/competitions/mashtots-dataset):
39 букв армянского алфавита в двух начертаниях, изображения 64×64 в градациях
серого. Обучение, оценка на отложенном тесте, разбор ошибок, предсказание для
одного файла и `submission.csv`.

Данные ожидаются в `data/mashtots/Train/<номер класса>/*.png` — как их скачать,
написано в README. Вариант этого же решения для запуска в Kaggle —
`mashtots_kaggle.ipynb`.

## 1. Импорты и конфигурация

Параметры переопределяются переменными окружения — удобно для быстрого прогона:
`MASHTOTS_EPOCHS=2 MASHTOTS_MAX_PER_CLASS=20 jupyter nbconvert --execute ...`.

In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
IMG_SIZE = 64            # родное разрешение датасета, апскейл не нужен
BATCH_SIZE = 128
VAL_FRACTION = 0.10
TEST_FRACTION = 0.10     # честная оценка: ни в обучении, ни в выборе модели не участвует
EPOCHS = int(os.environ.get("MASHTOTS_EPOCHS", 30))
DATA_DIR = Path(os.environ.get("MASHTOTS_DATA", "data/mashtots"))
MAX_PER_CLASS = int(os.environ.get("MASHTOTS_MAX_PER_CLASS", 0)) or None
MODEL_PATH = Path("mashtots_cnn.keras")

keras.utils.set_random_seed(SEED)
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid")

print("tensorflow", tf.__version__, "| keras", keras.__version__)
print("устройства:", [d.device_type for d in tf.config.list_physical_devices()])

## 2. Загрузка данных

Классы и файлы перебираются отсортированными — порядок чтения воспроизводим.
Метка берётся как `int(имя папки)`: при сортировке по строкам папка `10` встала
бы сразу после `1`. Массив выделяется заранее и хранится в `uint8` — 70 060
изображений 64×64 занимают 274 МиБ против 1.07 ГиБ во `float32`; нормализацию
делает слой `Rescaling` внутри модели. Нечитаемые файлы и посторонние
расширения пропускаются, а не роняют цикл.

In [ ]:
IMAGE_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".pgm"}


def list_class_dirs(root: Path) -> list[Path]:
    return sorted(
        (p for p in root.iterdir() if p.is_dir() and p.name.isdigit()),
        key=lambda p: int(p.name),
    )


def list_images(directory: Path) -> list[Path]:
    return sorted(p for p in directory.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXT)


def find_class_root(base: Path) -> Path:
    """Каталог, внутри которого лежат папки-классы с числовыми именами."""
    candidates = (base, base / "Train", base / "Train" / "Train",
                  Path("Train"), Path("Train") / "Train")
    for cand in dict.fromkeys(candidates):
        if cand.is_dir() and len(list_class_dirs(cand)) > 1:
            return cand
    raise FileNotFoundError(
        f"не найдены папки-классы. Распакуйте датасет соревнования так, чтобы "
        f"существовал путь {base}/Train/<номер класса>/*.png"
    )


def load_dataset(root: Path, img_size: int = IMG_SIZE, max_per_class: int | None = None):
    class_dirs = list_class_dirs(root)
    samples = []
    for cdir in class_dirs:
        samples += [(f, int(cdir.name)) for f in list_images(cdir)[:max_per_class]]

    X = np.empty((len(samples), img_size, img_size, 1), dtype=np.uint8)
    y = np.empty(len(samples), dtype=np.int16)

    n, resized, skipped = 0, 0, 0
    for path, label in samples:
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            skipped += 1
            continue
        if img.shape != (img_size, img_size):
            img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
            resized += 1
        X[n, :, :, 0] = img
        y[n] = label
        n += 1

    print(f"классов: {len(class_dirs)} | загружено: {n} | ресайз: {resized} | пропущено: {skipped}")
    return X[:n], y[:n]


CLASS_ROOT = find_class_root(DATA_DIR)
print("каталог классов:", CLASS_ROOT)

X, y = load_dataset(CLASS_ROOT, max_per_class=MAX_PER_CLASS)
NUM_CLASSES = int(y.max()) + 1
print(f"X: {X.shape} {X.dtype} ({X.nbytes / 2**20:.1f} MiB) | классов: {NUM_CLASSES}")

## 3. Обзор данных

Фон чёрный (`0`), штрих светлый. Отсюда решение по аугментации: пустоту после
сдвига и поворота надо заливать нулями, а дефолтный `fill_mode="reflect"`
затащил бы в кадр куски штриха.

In [ ]:
counts = pd.Series(y).value_counts().sort_index()
ratio = counts.max() / counts.min()
print(f"изображений на класс: min={counts.min()}, median={int(counts.median())}, max={counts.max()}")
print(f"дисбаланс max/min = {ratio:.2f}  ->  "
      f"{'веса классов не нужны' if ratio < 1.5 else 'стоит рассмотреть class_weight'}")

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
axes[0].bar(counts.index, counts.values, width=1.0)
axes[0].set(title="Число изображений по классам", xlabel="класс", ylabel="кол-во")
axes[1].hist(X[:: max(1, len(X) // 500)].ravel(), bins=50)
axes[1].set(title="Распределение значений пикселей", xlabel="значение", yscale="log")
plt.tight_layout()
plt.show()

In [ ]:
sample_idx = rng.choice(len(X), size=min(24, len(X)), replace=False)

fig, axes = plt.subplots(3, 8, figsize=(12, 5))
for ax, i in zip(axes.ravel(), sample_idx):
    ax.imshow(X[i, :, :, 0], cmap="gray")
    ax.set_title(f"class {y[i]}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Разбиение train / val / test

Стратифицированно, 80 / 10 / 10. `val` нужен для early stopping и планировщика
LR, `test` трогаем один раз в конце: по нему не принимается ни одно решение о
модели.

In [ ]:
def stratify_or_none(labels: np.ndarray, test_size: float):
    """Метки для stratify, либо None: стратификация требует минимум 2 примера на класс."""
    counts = np.bincount(labels)
    counts = counts[counts > 0]
    n_test = int(np.floor(test_size * len(labels)))
    if counts.min() >= 2 and n_test >= len(counts) and len(labels) - n_test >= len(counts):
        return labels
    print(f"стратификация отключена: примеров на класс минимум {counts.min()}, "
          f"классов {len(counts)} — увеличьте MASHTOTS_MAX_PER_CLASS")
    return None


hold_fraction = VAL_FRACTION + TEST_FRACTION
test_share_of_hold = TEST_FRACTION / hold_fraction

X_train, X_hold, y_train, y_hold = train_test_split(
    X, y, test_size=hold_fraction, random_state=SEED,
    stratify=stratify_or_none(y, hold_fraction),
)
X_val, X_test, y_val, y_test = train_test_split(
    X_hold, y_hold, test_size=test_share_of_hold, random_state=SEED,
    stratify=stratify_or_none(y_hold, test_share_of_hold),
)
del X_hold, y_hold

for name, a, b in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"{name:<6} {len(a):>7} изображений | классов: {len(np.unique(b))}")

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def make_ds(images: np.ndarray, labels: np.ndarray, training: bool = False) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices((images, labels.astype("int32")))
    if training:
        ds = ds.shuffle(min(len(images), 10_000), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_ds(X_train, y_train, training=True)
val_ds = make_ds(X_val, y_val)
test_ds = make_ds(X_test, y_test)

## 5. Модель

* `Rescaling` и аугментация — слоями внутри модели: нормализация применяется и
  на инференсе, а `Random*`-слои Keras сами отключаются вне обучения.
* Без отражений — буквы зеркально несимметричны, флип превратил бы часть классов
  в мусор. Только небольшие поворот, сдвиг и зум, пустота заливается нулями.
* Три блока `Conv-BN-ReLU ×2 → MaxPool → Dropout`: пара свёрток 3×3 даёт
  рецептивное поле 5×5 дешевле, чем одна 5×5.
* Голова без сужений: `Flatten(8·8·128) → Dense(256) → BN → Dropout → Dense(78)`,
  всего около 2.5 М параметров.

In [ ]:
# при дефолтном momentum=0.99 скользящие статистики BN сходятся только к ~2000
# шагам, и всё это время val-метрики стоят на 1/78 при растущем val-loss
BN_MOMENTUM = 0.9


def build_model(img_size: int = IMG_SIZE, num_classes: int = NUM_CLASSES) -> keras.Model:
    inputs = keras.Input(shape=(img_size, img_size, 1), name="image")

    x = layers.Rescaling(1.0 / 255)(inputs)
    x = layers.RandomRotation(0.03, fill_mode="constant", fill_value=0.0)(x)  # доля от 360°, то есть ±10.8°
    x = layers.RandomTranslation(0.08, 0.08, fill_mode="constant", fill_value=0.0)(x)
    x = layers.RandomZoom(0.10, fill_mode="constant", fill_value=0.0)(x)

    for filters in (32, 64, 128):
        for _ in range(2):
            x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization(momentum=BN_MOMENTUM)(x)
            x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization(momentum=BN_MOMENTUM)(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="probs")(x)

    return keras.Model(inputs, outputs, name="mashtots_cnn")


model = build_model()
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3")],
)
model.summary()

## 6. Обучение

`EarlyStopping(restore_best_weights=True)` возвращает лучшие веса, а не
последние; `ReduceLROnPlateau` уменьшает LR вдвое на плато — основной прирост
приходится на конец обучения; `ModelCheckpoint` пишет лучшую модель на диск.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=6, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        MODEL_PATH, monitor="val_accuracy", save_best_only=True, verbose=0
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    shuffle=False,  # перемешивание уже делает train_ds.shuffle(...)
)

In [ ]:
hist = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
hist[["loss", "val_loss"]].plot(ax=axes[0], title="Loss")
axes[0].axhline(np.log(NUM_CLASSES), ls="--", c="r", label=f"ln({NUM_CLASSES}) — уровень «не учится»")
hist[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="Accuracy")
axes[1].axhline(1 / NUM_CLASSES, ls="--", c="r", label=f"1/{NUM_CLASSES} — случайное угадывание")
for ax in axes:
    ax.set_xlabel("эпоха")
    ax.legend()
plt.tight_layout()
plt.show()

print(f"лучшая эпоха по val_accuracy: {int(np.argmax(history.history['val_accuracy'])) + 1}"
      f" из {len(hist)}")

## 7. Оценка на отложенном тесте

In [ ]:
for name, value in model.evaluate(test_ds, verbose=0, return_dict=True).items():
    print(f"test {name:<10} {value:.4f}")

probs = model.predict(test_ds, verbose=0)
y_pred = probs.argmax(axis=1)

report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
per_class = (
    pd.DataFrame(report).T.loc[lambda d: d.index.str.isdigit()]
    .astype({"support": int})
    .sort_values("f1-score")
)
print("\n10 самых трудных классов (по f1):")
print(per_class.head(10).round(3))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=np.arange(NUM_CLASSES))

plt.figure(figsize=(11, 9))
sns.heatmap(cm, cmap="viridis", square=True, cbar_kws={"shrink": 0.7})
plt.title(f"Confusion matrix ({NUM_CLASSES} классов)")
plt.xlabel("предсказано")
plt.ylabel("истина")
plt.tight_layout()
plt.show()

off = cm.copy()
np.fill_diagonal(off, 0)
pairs = [(a, b) for a, b in np.dstack(
    np.unravel_index(np.argsort(off, axis=None)[::-1], off.shape))[0][:10] if off[a, b] > 0]

if not pairs:
    print("перепутанных пар нет")
else:
    print("Чаще всего путаются (истина -> предсказание, число случаев):")
    for true_c, pred_c in pairs:
        print(f"  {true_c:>2} -> {pred_c:>2} : {off[true_c, pred_c]}")

In [ ]:
wrong = np.flatnonzero(y_pred != y_test)
print(f"ошибок на тесте: {len(wrong)} из {len(y_test)}")

if len(wrong):
    worst = wrong[np.argsort(probs[wrong, y_pred[wrong]])[::-1][:16]]
    fig, axes = plt.subplots(2, 8, figsize=(13, 4))
    for ax, i in zip(axes.ravel(), worst):
        ax.imshow(X_test[i, :, :, 0], cmap="gray")
        ax.set_title(f"{y_test[i]} -> {y_pred[i]}\np={probs[i, y_pred[i]]:.2f}", fontsize=8)
        ax.axis("off")
    for ax in axes.ravel()[len(worst):]:
        ax.axis("off")
    fig.suptitle("Самые уверенные ошибки", y=1.04)
    plt.tight_layout()
    plt.show()

## 8. Предсказание для одного файла

`preprocess_image` — тот же путь предобработки, что и при обучении, и он же
используется для соревновательного теста ниже, поэтому расхождение между
обучением и инференсом исключено. Нормализация живёт внутри модели, поэтому
функция возвращает `uint8`.

In [ ]:
def preprocess_image(path, img_size: int = IMG_SIZE) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"не удалось прочитать изображение: {path}")
    if img.shape != (img_size, img_size):
        img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
    return img.reshape(1, img_size, img_size, 1)


def predict_letter(path, top_k: int = 3):
    # прямой вызов модели вместо .predict(): для одной картинки быстрее и не
    # провоцирует ретрейсинг tf.function на каждый новый размер батча
    p = np.asarray(model(preprocess_image(path), training=False))[0]
    top = np.argsort(p)[::-1][:top_k]
    return int(top[0]), [(int(c), float(p[c])) for c in top]


demo_path = next(
    p for p in list_images(list_class_dirs(CLASS_ROOT)[0])
    if cv2.imread(str(p), cv2.IMREAD_GRAYSCALE) is not None
)

cls, top = predict_letter(demo_path)
print(f"файл: {demo_path}  (истинный класс по имени папки: {demo_path.parent.name})")
print(f"предсказано: {cls}")
print("top-3:", ", ".join(f"{c} ({p:.3f})" for c, p in top))

## 9. Submission

В архиве соревнования тест лежит либо каталогом картинок, либо таблицей
`new_test.csv` с развёрнутыми пикселями 64×64 — поддерживаются оба варианта.
Имена колонок берутся из `sample_submission.csv`, когда он есть.

In [ ]:
def stack_images(paths) -> np.ndarray:
    """Батч из файлов, без промежуточного списка массивов (тест — это 50 тыс. картинок)."""
    out = np.empty((len(paths), IMG_SIZE, IMG_SIZE, 1), dtype=np.uint8)
    for i, path in enumerate(paths):
        out[i] = preprocess_image(path)[0]
    return out


def load_competition_test(base: Path):
    """(images uint8 (N,64,64,1), ids) для теста соревнования или None, если его нет."""
    for name in ("new_test", "Test", "test"):
        d = base / name
        if d.is_dir():
            files = sorted(p for p in d.rglob("*") if p.suffix.lower() in IMAGE_EXT)
            if files:
                return stack_images(files), [p.stem for p in files]

    for csv_path in (base / "new_test.csv", base / "test.csv"):
        if not csv_path.is_file():
            continue
        df = pd.read_csv(csv_path)
        num = df.select_dtypes(include="number")
        pixel_cols = [c for c in num.columns if num[c].between(0, 255).all()]
        if len(pixel_cols) >= IMG_SIZE * IMG_SIZE:
            # развёрнутые пиксели; берём последние 64*64 колонок, если первая — это id
            pix = num[pixel_cols[-IMG_SIZE * IMG_SIZE:]].to_numpy(dtype=np.uint8)
            imgs = pix.reshape(-1, IMG_SIZE, IMG_SIZE, 1)
            ids = (df[df.columns[0]].tolist() if len(pixel_cols) < len(df.columns)
                   else list(range(len(df))))
            return imgs, ids
        for col in df.columns:
            if df[col].astype(str).str.contains(r"\.(?:png|jpg|jpeg|bmp)$", case=False).any():
                return (stack_images([csv_path.parent / p for p in df[col]]),
                        df[df.columns[0]].tolist())
    return None


test_data = load_competition_test(DATA_DIR)

if test_data is None:
    print("тест соревнования не найден — пропускаем submission")
else:
    imgs, ids = test_data
    preds = model.predict(imgs, batch_size=BATCH_SIZE, verbose=0)

    sample = DATA_DIR / "sample_submission.csv"
    id_col, target_col = ("Id", "Category")
    if sample.is_file():
        id_col, target_col = pd.read_csv(sample, nrows=0).columns[:2]

    submission = pd.DataFrame({id_col: ids, target_col: preds.argmax(axis=1)})
    submission.to_csv("submission.csv", index=False)
    print(f"submission.csv: {len(submission)} строк, колонки {list(submission.columns)}")
    print(f"средняя уверенность: {preds.max(axis=1).mean():.3f}")
    display(submission.head())